In [3]:
# Notebook: Modeling — training, CV, hyperparameter search, and artifact saving
# Sections: reproducible split → pipeline wrap → baseline training → randomized search → save model

# Ensure src/ is on the Python path for notebook imports
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.inspection import permutation_importance
import joblib

from src.data import load_california
from src.features import df_to_Xy, get_numeric_features, build_preprocessor, get_expanded_feature_names
from src.models import get_default_models, evaluate_models

ROOT = Path("..").resolve()
np.random.seed(42)

# Load data
df = load_california(sample_n=2000)
X, y = df_to_Xy(df)

# Stratified split by binned income (recommended for California)
df["_inc_bin"] = pd.qcut(df["MedInc"], q=5, duplicates="drop")
train_idx, test_idx = train_test_split(df.index, test_size=0.2, random_state=42, stratify=df["_inc_bin"])
X_train, y_train = X.loc[train_idx], y.loc[train_idx]
X_test, y_test = X.loc[test_idx], y.loc[test_idx]

# Build pipeline
num_feats = get_numeric_features(X)
pre = build_preprocessor(numeric_features=num_feats)

# Baseline evaluation (Ridge, RandomForest)
models = get_default_models()
cv_results = evaluate_models(pre, X_train, y_train, models=models, cv=5)
print("CV results (train):", cv_results)

# Example: RandomizedSearch for RandomForest
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from scipy.stats import randint

pipe = Pipeline([("pre", pre), ("model", RandomForestRegressor(random_state=42, n_jobs=-1))])
param_dist = {"model__n_estimators": randint(100, 800), "model__max_depth": randint(3, 20), "model__max_features": ["sqrt", "log2", None]}
rs = RandomizedSearchCV(pipe, param_distributions=param_dist, n_iter=12, scoring="neg_mean_squared_error", cv=3, random_state=42, n_jobs=-1)
rs.fit(X_train, y_train)
print("Best params:", rs.best_params_)

# Evaluate on test
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
best = rs.best_estimator_
preds = best.predict(X_test)
# compute RMSE robustly (avoid relying on 'squared' keyword which may not exist in older sklearn)
rmse = float(np.sqrt(mean_squared_error(y_test, preds)))
mae = float(mean_absolute_error(y_test, preds))
r2 = float(r2_score(y_test, preds))
print(f"Test RMSE={rmse:.4f}, MAE={mae:.4f}, R2={r2:.4f}")

# Permutation importance (with expanded feature names)
feature_names = get_expanded_feature_names(best.named_steps["pre"], list(X_test.columns))
pi = permutation_importance(best, X_test, y_test, n_repeats=12, random_state=42, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=feature_names).sort_values(ascending=False).head(10)
print("Top features (permutation)")
print(imp)

# Save artifact
(Path("..") / "models").mkdir(exist_ok=True)
joblib.dump(best, Path("..") / "models" / "best_pipeline.pkl")
print("Saved pipeline to models/best_pipeline.pkl")

# Save experiment summary (with top features)
exp = {"best_params": rs.best_params_, "test_metrics": {"rmse": rmse, "mae": mae, "r2": r2}, "top_features": imp.index.tolist()}
with open(Path("..") / "configs" / "model_summary.json", "w", encoding="utf-8") as f:
    json.dump(exp, f, indent=2)
print("Wrote configs/model_summary.json")

CV results (train): {'ridge': {'rmse': 0.9209706508232515, 'fold_scores': [0.5203298075262517, 0.512623159210011, 0.5914164436503923, 0.5436913791751692, 2.7594091229164674]}, 'rf': {'rmse': 0.6334539298759789, 'fold_scores': [0.375412278380138, 0.39383234305285814, 0.36691055777025205, 0.44814798245705684, 0.4248824425644343]}}
Best params: {'model__max_depth': 17, 'model__max_features': 'log2', 'model__n_estimators': 662}
Test RMSE=0.5694, MAE=0.3948, R2=0.7273
Top features (permutation)
num__MedInc        0.596988
num__Latitude      0.196865
num__Longitude     0.160062
num__AveOccup      0.159355
num__AveRooms      0.057528
num__HouseAge      0.040384
num__AveBedrms     0.010881
num__Population   -0.003369
dtype: float64
Saved pipeline to models/best_pipeline.pkl
Wrote configs/model_summary.json
